<a href="https://colab.research.google.com/github/DanielHashmi/Homework_Python_Projects/blob/main/25%20Projects/Connect_Four_Game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import numpy as np
import pygame
import sys
import math

BLUE = (52, 152, 219)
BLACK = (44, 62, 80)
RED = (231, 76, 60)
YELLOW = (241, 196, 15)
WHITE = (236, 240, 241)
GREEN = (46, 204, 113)
DARK_GREEN = (39, 174, 96)
LIGHT_GRAY = (149, 165, 166)

ROW_COUNT = 6
COLUMN_COUNT = 7

SQUARE_SIZE = 100
RADIUS = int(SQUARE_SIZE/2 - 5)

def create_board():
    return np.zeros((ROW_COUNT, COLUMN_COUNT))

def drop_piece(board, row, col, piece):
    board[row][col] = piece

def is_valid_location(board, col):
    return board[ROW_COUNT-1][col] == 0

def get_next_open_row(board, col):
    for r in range(ROW_COUNT):
        if board[r][col] == 0:
            return r
    return -1

def winning_move(board, piece):
    for c in range(COLUMN_COUNT-3):
        for r in range(ROW_COUNT):
            if board[r][c] == piece and board[r][c+1] == piece and board[r][c+2] == piece and board[r][c+3] == piece:
                return True

    for c in range(COLUMN_COUNT):
        for r in range(ROW_COUNT-3):
            if board[r][c] == piece and board[r+1][c] == piece and board[r+2][c] == piece and board[r+3][c] == piece:
                return True

    for c in range(COLUMN_COUNT-3):
        for r in range(ROW_COUNT-3):
            if board[r][c] == piece and board[r+1][c+1] == piece and board[r+2][c+2] == piece and board[r+3][c+3] == piece:
                return True

    for c in range(COLUMN_COUNT-3):
        for r in range(3, ROW_COUNT):
            if board[r][c] == piece and board[r-1][c+1] == piece and board[r-2][c+2] == piece and board[r-3][c+3] == piece:
                return True

    return False

def is_board_full(board):
    return not np.any(board == 0)

def draw_board(board, screen):
    for c in range(COLUMN_COUNT):
        for r in range(ROW_COUNT):
            pygame.draw.rect(screen, BLUE, (c*SQUARE_SIZE, r*SQUARE_SIZE+SQUARE_SIZE, SQUARE_SIZE, SQUARE_SIZE))
            pygame.draw.circle(screen, BLACK, (int(c*SQUARE_SIZE+SQUARE_SIZE/2), int(r*SQUARE_SIZE+SQUARE_SIZE+SQUARE_SIZE/2)), RADIUS)

    for c in range(COLUMN_COUNT):
        for r in range(ROW_COUNT):
            if board[r][c] == 1:
                pygame.draw.circle(screen, RED, (int(c*SQUARE_SIZE+SQUARE_SIZE/2), height - int(r*SQUARE_SIZE+SQUARE_SIZE/2)), RADIUS)
            elif board[r][c] == 2:
                pygame.draw.circle(screen, YELLOW, (int(c*SQUARE_SIZE+SQUARE_SIZE/2), height - int(r*SQUARE_SIZE+SQUARE_SIZE/2)), RADIUS)

    pygame.display.update()

def draw_fancy_button(screen, font, mouse_pos=None, is_clicking=False):
    button_width = 220
    button_height = 60
    button_x = (width - button_width) // 2
    button_y = height - 80

    button_rect = pygame.Rect(button_x, button_y, button_width, button_height)

    shadow_rect = pygame.Rect(button_x + 4, button_y + 4, button_width, button_height)
    pygame.draw.rect(screen, (50, 50, 50, 128), shadow_rect, border_radius=15)

    button_color = GREEN
    text_offset = 0

    if mouse_pos and button_rect.collidepoint(mouse_pos):
        if is_clicking:
            button_color = DARK_GREEN
            text_offset = 2
        else:
            button_color = (86, 185, 90)

    pygame.draw.rect(screen, button_color, button_rect, border_radius=15)

    highlight_rect = pygame.Rect(button_x + 3, button_y + 3, button_width - 6, 5)
    pygame.draw.rect(screen, (120, 220, 120, 128), highlight_rect, border_radius=5)

    retry_text = font.render("Play Again", True, BLACK)
    text_shadow = font.render("Play Again", True, (40, 40, 40))

    text_x = button_x + (button_width - retry_text.get_width()) // 2
    text_y = button_y + (button_height - retry_text.get_height()) // 2

    screen.blit(text_shadow, (text_x + 2, text_y + 2 + text_offset))
    screen.blit(retry_text, (text_x, text_y + text_offset))

    pygame.display.update()

    return button_rect

def main():
    pygame.init()

    global width, height
    width = COLUMN_COUNT * SQUARE_SIZE
    height = (ROW_COUNT + 1) * SQUARE_SIZE
    size = (width, height)
    screen = pygame.display.set_mode(size)
    pygame.display.set_caption('Connect Four')

    game_font = pygame.font.SysFont("Arial", 60, bold=True)
    button_font = pygame.font.SysFont("Arial", 30, bold=True)

    running = True
    while running:
        board = create_board()
        game_over = False
        turn = 0

        screen.fill(BLACK)
        draw_board(board, screen)
        pygame.display.update()

        while not game_over:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    pygame.quit()
                    sys.exit()

                if event.type == pygame.MOUSEMOTION and not game_over:
                    pygame.draw.rect(screen, BLACK, (0, 0, width, SQUARE_SIZE))
                    posx = event.pos[0]
                    if turn == 0:
                        pygame.draw.circle(screen, RED, (posx, int(SQUARE_SIZE/2)), RADIUS)
                    else:
                        pygame.draw.circle(screen, YELLOW, (posx, int(SQUARE_SIZE/2)), RADIUS)
                    pygame.display.update()

                if event.type == pygame.MOUSEBUTTONDOWN and not game_over:
                    pygame.draw.rect(screen, BLACK, (0, 0, width, SQUARE_SIZE))

                    posx = event.pos[0]
                    col = int(math.floor(posx/SQUARE_SIZE))

                    if 0 <= col < COLUMN_COUNT and is_valid_location(board, col):
                        row = get_next_open_row(board, col)
                        drop_piece(board, row, col, turn + 1)

                        if winning_move(board, turn + 1):
                            pygame.draw.rect(screen, BLACK, (0, 0, width, SQUARE_SIZE))
                            winner_color = RED if turn == 0 else YELLOW
                            label = game_font.render(f"Player {turn + 1} wins!!", True, winner_color)
                            screen.blit(label, ((width - label.get_width()) // 2, 10))
                            game_over = True

                        elif is_board_full(board):
                            pygame.draw.rect(screen, BLACK, (0, 0, width, SQUARE_SIZE))
                            tie_label = game_font.render("Tie Game!", True, BLUE)
                            screen.blit(tie_label, ((width - tie_label.get_width()) // 2, 10))
                            game_over = True

                        turn = (turn + 1) % 2

                    draw_board(board, screen)

            if game_over:
                button_rect = draw_fancy_button(screen, button_font)
                waiting_for_click = True

                while waiting_for_click:
                    mouse_pos = pygame.mouse.get_pos()
                    mouse_clicked = pygame.mouse.get_pressed()[0]

                    button_rect = draw_fancy_button(screen, button_font, mouse_pos, mouse_clicked)

                    for event in pygame.event.get():
                        if event.type == pygame.QUIT:
                            pygame.quit()
                            sys.exit()
                        if event.type == pygame.MOUSEBUTTONUP:
                            if button_rect.collidepoint(event.pos):
                                draw_fancy_button(screen, button_font, event.pos, True)
                                pygame.time.wait(150)

                                waiting_for_click = False
                                break

                    pygame.time.wait(10)

if __name__ == "__main__":
    main()